In [2]:
import re
import pdfplumber
from pathlib import Path

FOLDER_PATH = r"C:\Users\loyd5\Desktop\modularized\INVOICES\LOYD BATCH 1"
DRY_RUN = False

STOPWORDS = {
    "dear", "sir", "madam", "management", "team", "staff", "all",
    "whom", "may", "concern", "the", "our", "your", "their",
    "hiring", "manager", "director", "human", "resources",
    "principal", "thro", "ref", "date", "nairobi", "kenya",
    "re", "attention", "attn", "to", "cc", "bcc",
}

INSTITUTION_WORDS = {
    "polytechnic", "university", "college", "institute", "school",
    "national", "Technical and Training Institute","Vocational and Training Institute","county", "government", "ministry", "department",
    "hospital", "clinic", "centre", "center", "authority", "board",
    "council", "commission", "corporation", "company", "limited",
}

def looks_like_name(line):
    line = line.strip()
    if not line or any(c.isdigit() for c in line):
        return False
    words = line.split()
    if not (2 <= len(words) <= 4):
        return False
    if not all(w[0].isupper() for w in words if w):
        return False
    lower_words = {w.lower() for w in words}
    if lower_words & STOPWORDS:
        return False
    if lower_words & INSTITUTION_WORDS:
        return False
    return True

def extract_name(text):
    lines = [l.strip() for l in text.splitlines()]
    search_lines = lines[:50]
    search_block = "\n".join(search_lines)

    # Strategy 1: Dear salutation
    for pattern in [
        r"Dear\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss|Dr\.?|Prof\.?)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})",
        r"Dear\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s*[,:]",
    ]:
        m = re.search(pattern, search_block)
        if m and looks_like_name(m.group(1)):
            return m.group(1).strip()

    # Strategy 2: To/Attn line
    m = re.search(r"(?:To|Attn\.?|Attention)\s*[:\-]\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})", search_block)
    if m and looks_like_name(m.group(1)):
        return m.group(1).strip()

    # Strategy 3: Name after Ref No / Date block (TVETCDACC format)
    ref_pattern = re.compile(r"(?:Ref\.?\s*No\.?|Date\s*:)", re.IGNORECASE)
    for i, line in enumerate(search_lines):
        if ref_pattern.search(line):
            for candidate in search_lines[i + 1: i + 6]:
                if looks_like_name(candidate):
                    return candidate.strip()

    # Strategy 4: Fallback — first name-looking line
    for line in search_lines:
        if looks_like_name(line):
            return line

    return None

def sanitize(name):
    return re.sub(r"\s+", " ", re.sub(r'[\\/*?:"<>|]', "", name)).strip()

def unique_path(folder, stem):
    p = folder / f"{stem}.pdf"
    n = 2
    while p.exists():
        p = folder / f"{stem} ({n}).pdf"
        n += 1
    return p

def process(folder, dry_run=False):
    folder = Path(folder)
    pdfs = sorted(folder.glob("*.pdf"))
    print(f"\n{len(pdfs)} PDF(s) found | {'DRY RUN' if dry_run else 'LIVE'}\n" + "-"*50)
    renamed = skipped = errors = 0

    for pdf in pdfs:
        print(f"\n{pdf.name}")
        try:
            with pdfplumber.open(pdf) as f:
                text = "\n".join(p.extract_text() or "" for p in f.pages)
        except Exception as e:
            print(f"  ERROR: {e}"); errors += 1; continue

        if not text.strip():
            print("  SKIP: no text extracted"); skipped += 1; continue

        name = extract_name(text)
        if not name:
            print("  SKIP: name not found")
            print("  Text preview:", text[:300].replace("\n", " ↵ "))
            skipped += 1; continue

        new_path = unique_path(folder, sanitize(name))
        print(f"  → {new_path.name}")

        if dry_run:
            print("  DRY RUN — no change"); renamed += 1; continue

        try:
            pdf.rename(new_path); print("  OK"); renamed += 1
        except Exception as e:
            print(f"  ERROR: {e}"); errors += 1

    print(f"\n{'='*50}\nRenamed: {renamed} | Skipped: {skipped} | Errors: {errors}")

if __name__ == "__main__":
    process(FOLDER_PATH, dry_run=DRY_RUN)


110 PDF(s) found | LIVE
--------------------------------------------------

doc09823420251127103459_115.pdf
  SKIP: no text extracted

doc09823520251127103958_110.pdf
  SKIP: no text extracted

doc09823520251127103958_111.pdf
  SKIP: no text extracted

doc09823520251127103958_112.pdf
  SKIP: no text extracted

doc09823520251127103958_113.pdf
  SKIP: no text extracted

doc09823520251127103958_114.pdf
  SKIP: no text extracted

doc09823520251127103958_115.pdf
  SKIP: no text extracted

doc09823520251127103958_116.pdf
  SKIP: no text extracted

doc09823520251127103958_117.pdf
  SKIP: no text extracted

doc09823520251127103958_118.pdf
  SKIP: no text extracted

doc09823520251127103958_119.pdf
  SKIP: no text extracted

doc09823520251127103958_120.pdf
  SKIP: no text extracted

doc09823520251127103958_121.pdf
  SKIP: no text extracted

doc09823520251127103958_122.pdf
  SKIP: no text extracted

doc09823520251127103958_123.pdf
  SKIP: no text extracted

doc09823620251127105143_001.pdf
  SKIP

In [9]:
import re
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

# ── SET THESE PATHS ───────────────────────────────────────────────────────────
FOLDER_PATH   = r"C:\Users\loyd5\Desktop\modularized\INVOICES\LOYD BATCH 1"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH  = r"C:\Users\loyd5\Desktop\Learning Python\Automated Letters Sending\Release-24.07.0-0\poppler-24.07.0\Library\bin"
DRY_RUN       = False
# ─────────────────────────────────────────────────────────────────────────────

pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

STOPWORDS = {
    "dear", "sir", "madam", "management", "team", "staff", "all",
    "whom", "may", "concern", "the", "our", "your", "their",
    "hiring", "manager", "director", "human", "resources",
    "principal", "thro", "ref", "date", "nairobi", "kenya",
    "re", "attention", "attn", "to", "cc", "bcc",
}

INSTITUTION_WORDS = {
    "polytechnic", "university", "college", "institute", "school",
    "national", "county", "government", "ministry", "department",
    "hospital", "clinic", "centre", "center", "authority", "board",
    "council", "commission", "corporation", "company", "limited",
}

def looks_like_name(line):
    line = line.strip()
    if not line or any(c.isdigit() for c in line):
        return False
    words = line.split()
    if not (2 <= len(words) <= 4):
        return False
    if not all(w[0].isupper() for w in words if w):
        return False
    lower_words = {w.lower() for w in words}
    if lower_words & STOPWORDS:
        return False
    if lower_words & INSTITUTION_WORDS:
        return False
    return True

def extract_name(text):
    lines = [l.strip() for l in text.splitlines()]
    search_lines = lines[:50]
    search_block = "\n".join(search_lines)

    # Strategy 1: Dear salutation
    for pattern in [
        r"Dear\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss|Dr\.?|Prof\.?)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})",
        r"Dear\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s*[,:]",
    ]:
        m = re.search(pattern, search_block)
        if m and looks_like_name(m.group(1)):
            return m.group(1).strip()

    # Strategy 2: To/Attn line
    m = re.search(r"(?:To|Attn\.?|Attention)\s*[:\-]\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})", search_block)
    if m and looks_like_name(m.group(1)):
        return m.group(1).strip()

    # Strategy 3: Name after Ref No / Date (TVETCDACC letter format)
    ref_pattern = re.compile(r"(?:Ref\.?\s*No\.?|Date\s*:)", re.IGNORECASE)
    for i, line in enumerate(search_lines):
        if ref_pattern.search(line):
            for candidate in search_lines[i + 1: i + 6]:
                if looks_like_name(candidate):
                    return candidate.strip()

    # Strategy 4: Fallback — first name-looking line
    for line in search_lines:
        if looks_like_name(line):
            return line

    return None

def sanitize(name):
    return re.sub(r"\s+", " ", re.sub(r'[\\/*?:"<>|]', "", name)).strip()

def unique_path(folder, stem):
    p = folder / f"{stem}.pdf"
    n = 2
    while p.exists():
        p = folder / f"{stem} ({n}).pdf"
        n += 1
    return p

def process(folder, dry_run=False):
    folder = Path(folder)
    pdfs = sorted(folder.glob("*.pdf"))
    print(f"\n{len(pdfs)} PDF(s) found | {'DRY RUN' if dry_run else 'LIVE'}\n" + "-"*50)
    renamed = skipped = errors = 0

    for pdf in pdfs:
        print(f"\n{pdf.name}")

        # OCR: convert PDF pages to images, then extract text
        try:
            images = convert_from_path(str(pdf), dpi=300, poppler_path=POPPLER_PATH)
            text = ""
            for img in images[:2]:  # Only first 2 pages needed
                text += pytesseract.image_to_string(img) + "\n"
        except Exception as e:
            print(f"  ERROR: {e}"); errors += 1; continue

        if not text.strip():
            print("  SKIP: no text extracted"); skipped += 1; continue

        name = extract_name(text)
        if not name:
            print("  SKIP: name not found")
            print("  Text preview:", text[:300].replace("\n", " ↵ "))
            skipped += 1; continue

        new_path = unique_path(folder, sanitize(name))
        print(f"  → {new_path.name}")

        if dry_run:
            print("  DRY RUN — no change"); renamed += 1; continue

        try:
            pdf.rename(new_path); print("  OK"); renamed += 1
        except Exception as e:
            print(f"  ERROR: {e}"); errors += 1

    print(f"\n{'='*50}\nRenamed: {renamed} | Skipped: {skipped} | Errors: {errors}")

if __name__ == "__main__":
    
    process(FOLDER_PATH, dry_run=DRY_RUN)


110 PDF(s) found | LIVE
--------------------------------------------------

Abraham Ntonja Nthuka (2).pdf
  → Abraham Ntonja Nthuka.pdf
  OK

Akach Wicklife (2).pdf
  → Akach Wicklife.pdf
  OK

Aloice Omondi (2).pdf
  → Aloice Omondi.pdf
  OK

Amisi Malach Nyang'au (2).pdf
  → Amisi Malach Nyang'au.pdf
  OK

Ann Gachuhi (2).pdf
  → Ann Gachuhi.pdf
  OK

Annah Komu (2).pdf
  → Annah Komu.pdf
  OK

Antony Kimani Ng'ang'a (2).pdf
  → Antony Kimani Ng'ang'a.pdf
  OK

Beatrice Karanja (2).pdf
  → Beatrice Karanja.pdf
  OK

Beatrice Muthoni (2).pdf
  → Beatrice Muthoni.pdf
  OK

Becky Chepkwony (2).pdf
  → Becky Chepkwony.pdf
  OK

Benson Otieno Odero (2).pdf
  → Benson Otieno Odero.pdf
  OK

Bernice Waithera Kimani (2).pdf
  → Bernice Waithera Kimani.pdf
  OK

Betty Kendi (2).pdf
  → Betty Kendi.pdf
  OK

Brian Wafula (2).pdf
  → Brian Wafula.pdf
  OK

Bruno Kisia (2).pdf
  → Bruno Kisia.pdf
  OK

Caroline Chebet Bor (2).pdf
  → Caroline Chebet Bor.pdf
  OK

Chepkwony Peter (2).pdf
  → Che

In [11]:
import re
import csv
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
FOLDER_PATH   = r"C:\Users\loyd5\Desktop\modularized\INVOICES\LOYD BATCH 1"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH  = r"C:\Users\loyd5\Desktop\Learning Python\Automated Letters Sending\Release-24.07.0-0\poppler-24.07.0\Library\bin"
OUTPUT_CSV    = r"C:\Users\loyd5\Desktop\recipients.csv"
# ─────────────────────────────────────────────────────────────────────────────

pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

STOPWORDS = {
    "dear", "sir", "madam", "management", "team", "staff", "all",
    "whom", "may", "concern", "the", "our", "your", "their",
    "hiring", "manager", "director", "human", "resources",
    "principal", "thro", "ref", "date", "nairobi", "kenya",
    "re", "attention", "attn", "to", "cc", "bcc",
}
INSTITUTION_WORDS = {
    "polytechnic", "university", "college", "institute", "school",
    "national", "Technical and Training Institute","Vocational and Training Institute","county", "government", "ministry", "department",
    "hospital", "clinic", "centre", "center", "authority", "board",
    "council", "commission", "corporation", "company", "limited",
}

def looks_like_name(line):
    line = line.strip()
    if not line or any(c.isdigit() for c in line):
        return False
    words = line.split()
    if not (2 <= len(words) <= 4):
        return False
    if not all(w[0].isupper() for w in words if w):
        return False
    lower_words = {w.lower() for w in words}
    if lower_words & STOPWORDS:
        return False
    if lower_words & INSTITUTION_WORDS:
        return False
    return True

def extract_name(text):
    lines = [l.strip() for l in text.splitlines()]
    search_lines = lines[:50]
    search_block = "\n".join(search_lines)

    for pattern in [
        r"Dear\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss|Dr\.?|Prof\.?)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})",
        r"Dear\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s*[,:]",
    ]:
        m = re.search(pattern, search_block)
        if m and looks_like_name(m.group(1)):
            return m.group(1).strip()

    m = re.search(r"(?:To|Attn\.?|Attention)\s*[:\-]\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})", search_block)
    if m and looks_like_name(m.group(1)):
        return m.group(1).strip()

    ref_pattern = re.compile(r"(?:Ref\.?\s*No\.?|Date\s*:)", re.IGNORECASE)
    for i, line in enumerate(search_lines):
        if ref_pattern.search(line):
            for candidate in search_lines[i + 1: i + 6]:
                if looks_like_name(candidate):
                    return candidate.strip()

    for line in search_lines:
        if looks_like_name(line):
            return line
    return None

def extract_email(text):
    emails = re.findall(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}", text)
    filtered = [e for e in emails if "tvetcdacc" not in e.lower() and "info@" not in e.lower()]
    return filtered[0] if filtered else None

def process(folder, output_csv):
    folder = Path(folder)
    pdfs = sorted(folder.glob("*.pdf"))
    print(f"\n{len(pdfs)} PDF(s) found\n" + "-"*50)

    rows = []
    found = not_found = errors = 0

    for pdf in pdfs:
        print(f"\n{pdf.name}")

        try:
            images = convert_from_path(str(pdf), dpi=300, poppler_path=POPPLER_PATH)
            text = ""
            for img in images[:2]:
                text += pytesseract.image_to_string(img) + "\n"
        except Exception as e:
            print(f"  ERROR: {e}"); errors += 1; continue

        name  = extract_name(text)
        email = extract_email(text)

        print(f"  Name  : {name  or 'NOT FOUND'}")
        print(f"  Email : {email or 'NOT FOUND'}")

        rows.append({
            "File":     pdf.name,
            "Name":     name  or "",
            "Email":    email or "",
            "Status":   "Found" if email else "No Email"
        })

        if email:
            found += 1
        else:
            not_found += 1

    # Write CSV
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["File", "Name", "Email", "Status"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\n{'='*50}")
    print(f"CSV saved to: {output_csv}")
    print(f"Emails found: {found} | Not found: {not_found} | Errors: {errors}")

if __name__ == "__main__":
    process(FOLDER_PATH, OUTPUT_CSV)


110 PDF(s) found
--------------------------------------------------

Abraham Ntonja Nthuka.pdf
  Name  : Abraham Ntonja Nthuka
  Email : ntoja.nthuka@gmail.com

Akach Wicklife.pdf
  Name  : Akach Wicklife
  Email : agutuakach@gmail.com

Aloice Omondi.pdf
  Name  : Aloice Omondi
  Email : aloiceomondil6@gmail.com

Amisi Malach Nyang'au.pdf
  Name  : Amisi Malach Nyang'au
  Email : malachamisi2024@gmail.com

Ann Gachuhi.pdf
  Name  : Ann Gachuhi
  Email : NOT FOUND

Annah Komu.pdf
  Name  : Annah Komu
  Email : NOT FOUND

Antony Kimani Ng'ang'a.pdf
  Name  : Antony Kimani Ng'ang'a
  Email : NOT FOUND

Beatrice Karanja.pdf
  Name  : Beatrice Karanja
  Email : beatricekaranja04@gmail.com

Beatrice Muthoni.pdf
  Name  : Beatrice Muthoni
  Email : NOT FOUND

Becky Chepkwony.pdf
  Name  : Becky Chepkwony
  Email : NOT FOUND

Benson Otieno Odero.pdf
  Name  : Benson Otieno Odero
  Email : benaulis@gmail.com

Bernice Waithera Kimani.pdf
  Name  : Bernice Waithera Kimani
  Email : NOT FOUND

Be

In [12]:
import re
import csv
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
FOLDER_PATH   = r"C:\Users\loyd5\Desktop\modularized\INVOICES\LOYD BATCH 1"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH  = r"C:\Users\loyd5\Desktop\Learning Python\Automated Letters Sending\Release-24.07.0-0\poppler-24.07.0\Library\bin"
OUTPUT_CSV    = r"C:\Users\loyd5\Desktop\recipients.csv"
# ─────────────────────────────────────────────────────────────────────────────

pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

STOPWORDS = {
    "dear", "sir", "madam", "management", "team", "staff", "all",
    "whom", "may", "concern", "the", "our", "your", "their",
    "hiring", "manager", "director", "human", "resources",
    "principal", "thro", "ref", "date", "nairobi", "kenya",
    "re", "attention", "attn", "to", "cc", "bcc",
}
INSTITUTION_WORDS = {
    "polytechnic", "university", "college", "institute", "school",
    "national", "technical and training institute", "vocational and training institute",
    "county", "government", "ministry", "department",
    "hospital", "clinic", "centre", "center", "authority", "board",
    "council", "commission", "corporation", "company", "limited",
}

def looks_like_name(line):
    line = line.strip()
    if not line or any(c.isdigit() for c in line):
        return False
    words = line.split()
    if not (2 <= len(words) <= 4):
        return False
    if not all(w[0].isupper() for w in words if w):
        return False
    lower_words = {w.lower() for w in words}
    if lower_words & STOPWORDS:
        return False
    if lower_words & INSTITUTION_WORDS:
        return False
    return True

def extract_name(text):
    lines = [l.strip() for l in text.splitlines()]
    search_lines = lines[:50]
    search_block = "\n".join(search_lines)

    for pattern in [
        r"Dear\s+(?:Mr\.?|Mrs\.?|Ms\.?|Miss|Dr\.?|Prof\.?)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})",
        r"Dear\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})\s*[,:]",
    ]:
        m = re.search(pattern, search_block)
        if m and looks_like_name(m.group(1)):
            return m.group(1).strip()

    m = re.search(r"(?:To|Attn\.?|Attention)\s*[:\-]\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})", search_block)
    if m and looks_like_name(m.group(1)):
        return m.group(1).strip()

    ref_pattern = re.compile(r"(?:Ref\.?\s*No\.?|Date\s*:)", re.IGNORECASE)
    for i, line in enumerate(search_lines):
        if ref_pattern.search(line):
            for candidate in search_lines[i + 1: i + 6]:
                if looks_like_name(candidate):
                    return candidate.strip()

    for line in search_lines:
        if looks_like_name(line):
            return line
    return None

def extract_email(text):
    # Fix common OCR misreads before searching
    fixes = [
        (r'©',       '@'),
        (r'\[at\]',  '@'),
        (r'\(at\)',  '@'),
        (r' at ',    '@'),
        (r'\[dot\]', '.'),
        (r'\(dot\)', '.'),
        (r' dot ',   '.'),
        (r'@\s+',    '@'),
        (r'\s+@',    '@'),
        (r'\.\s+',   '.'),
    ]
    cleaned = text
    for pattern, replacement in fixes:
        cleaned = re.sub(pattern, replacement, cleaned, flags=re.IGNORECASE)

    # Broad email pattern to catch OCR variations
    emails = re.findall(
        r"[a-zA-Z0-9._%+\-]{2,}@[a-zA-Z0-9.\-]{2,}\.[a-zA-Z]{2,6}",
        cleaned
    )

    if not emails:
        return None

    # Filter out org/sender emails, keep recipient's
    org_keywords = ["tvetcdacc", "info@", "cdacc", "knec", "tveta"]
    recipient_emails = [e for e in emails if not any(k in e.lower() for k in org_keywords)]

    if recipient_emails:
        return recipient_emails[0].lower()

    return emails[0].lower()

def process(folder, output_csv):
    folder = Path(folder)
    pdfs = sorted(folder.glob("*.pdf"))
    print(f"\n{len(pdfs)} PDF(s) found\n" + "-"*50)

    rows = []
    found = not_found = errors = 0

    for pdf in pdfs:
        print(f"\n{pdf.name}")

        try:
            images = convert_from_path(str(pdf), dpi=300, poppler_path=POPPLER_PATH)
            text = ""
            for img in images[:2]:
                text += pytesseract.image_to_string(img) + "\n"
        except Exception as e:
            print(f"  ERROR: {e}"); errors += 1; continue

        # DEBUG: shows exactly how OCR reads lines containing email characters
        print("  [DEBUG] Lines with @ or similar:")
        for line in text.splitlines():
            if any(c in line for c in ["@", "©", " at "]):
                print(f"    >>> {repr(line)}")

        name  = extract_name(text)
        email = extract_email(text)

        print(f"  Name  : {name  or 'NOT FOUND'}")
        print(f"  Email : {email or 'NOT FOUND'}")

        rows.append({
            "File":   pdf.name,
            "Name":   name  or "",
            "Email":  email or "",
            "Status": "Found" if email else "No Email"
        })

        if email: found += 1
        else:     not_found += 1

    # Write CSV
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["File", "Name", "Email", "Status"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"\n{'='*50}")
    print(f"CSV saved to: {output_csv}")
    print(f"Emails found: {found} | Not found: {not_found} | Errors: {errors}")

if __name__ == "__main__":
    process(FOLDER_PATH, OUTPUT_CSV)


110 PDF(s) found
--------------------------------------------------

Abraham Ntonja Nthuka.pdf
  [DEBUG] Lines with @ or similar:
    >>> 'NAIROBI, KENYA Email: info@tvetcdacc.go.ke'
    >>> 'ntoja.nthuka@gmail.com'
    >>> 'assessment at the Eldoret National Polytechnic.'
    >>> 'be carried out at the marking centre are not to be disclosed to any other party that is'
    >>> 'You will be given accommodation at the marking center and where accommodation'
    >>> 'info@tvetcdacc.go.ke .'
  Name  : Abraham Ntonja Nthuka
  Email : ntoja.nthuka@gmail.com

Akach Wicklife.pdf
  [DEBUG] Lines with @ or similar:
    >>> 'NAIROBI, KENYA Email: info@tvetcdacc.go.ke'
    >>> 'agutuakach@gmail.com'
    >>> 'assessment at the Eldoret National Polytechnic.'
    >>> 'be carried out at the marking centre are not to be disclosed to any other party that is'
    >>> 'You will be given accommodation at the marking center and where accommodation'
    >>> 'info @tvetcdacc.go.ke .'
  Name  : Akach Wicklife

In [ ]:
import re
import csv
import smtplib
import pytesseract
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from pdf2image import convert_from_path
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
FOLDER_PATH    = r"C:\Users\loyd5\Desktop\modularized\INVOICES\LOYD BATCH 1"
TESSERACT_CMD  = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH   = r"C:\Users\loyd5\Desktop\Learning Python\Automated Letters Sending\Release-24.07.0-0\poppler-24.07.0\Library\bin"
OUTPUT_CSV     = r"C:\Users\loyd5\Desktop\recipients.csv"

SENDER_EMAIL    = "loyd5kinoti@gmail.com"
SENDER_PASSWORD = "xxxx xxxx xxxx xxxx"   # ← your Gmail App Password

DRY_RUN = False  # Set True to preview without sending

# ── EDIT YOUR SUBJECT AND MESSAGE HERE ────────────────────────────────────────



         EMAIL COMPOSER



📧 Enter email SUBJECT:
>  Invitation



📝 Enter email MESSAGE body.
   Use {name} anywhere to personalise with recipient's name.
   Type END on a new line when done.



 We welcome you {name}.
 Regards.
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 j


In [ ]:
import re
import csv
import smtplib
import pytesseract
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from pdf2image import convert_from_path
from pathlib import Path

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
FOLDER_PATH    = r"C:\Users\loyd5\Desktop\modularized\INVOICES\LOYD BATCH 1"
TESSERACT_CMD  = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
POPPLER_PATH   = r"C:\Users\loyd5\Desktop\Learning Python\Automated Letters Sending\Release-24.07.0-0\poppler-24.07.0\Library\bin"
OUTPUT_CSV     = r"C:\Users\loyd5\Desktop\recipients.csv"

SENDER_EMAIL    = "loyd5kinoti@gmail.com"
SENDER_PASSWORD = "xxxx xxxx xxxx xxxx"   # ← your Gmail App Password

DRY_RUN = False  # Set True to preview without sending

# ── EDIT YOUR SUBJECT AND MESSAGE HERE ────────────────────────────────────────

